In [1]:
import numpy as np

W1 = {
    0: {2, 4},
    1: {3, 4},
    2: {1},
    3: {0, 1, 4},
    4: set()}

W2 = {
    0: {1},
    1: {2},
    2: {0},
    3: {4},
    4: {5},
    5: {3}}

In [2]:
def surf_step(web):
    
    # Input: Et netværk som dictionary og en start side
    # Output: Sandsynlighedsfordeling som dictionary for næste hjemmeside
    
    distribution=dict() # laver en tom dictionary til at gemme sandsynlighedsfordelingen

    for page, links in web.items(): # for hver hjemmeside og dens links i web-dictionaryen
        if links == set(): # hvis hjemmesiden ikke har nogen links, så skal den have en ligelig fordeling over alle sider
            distribution[page] = [1/len(web) for i in range(len(web))] # fordel sandsynligheden ligeligt over alle sider fx laver den [0, 0, 0, 0, 0] -> [1/5, 1/5, 1/5, 1/5, 1/5]
        else: # hvis hjemmesiden har links, så skal den fordele sandsynligheden ligeligt over de sider den linker til
            distribution[page] = [0 for i in range(len(web))] # starter med at lave [0, 0, 0, 0, 0] for hver hjemmeside
            for link in links:
                distribution[page][link] += 1/len(links) # her taget den fx [0, 1, 0, 1, 0] og laver den til [0, 1/2, 0, 1/2, 0] hvis der er 2 links

    
    return distribution

In [3]:
def modified_link_matrix(web, pagelist=None, d=0.85):

    # Input: web (dictionary), pagelist (liste over nøgler), d (dæmpningsfaktor)
    # Output: d*A^T + (1-d)*E/N
    
    N=len(web)
    E_N = np.ones((N, N))
    distri = surf_step(web)
    L = np.array([i for i in distri.values()])

    # A: NxN numpy array, hvor række j har ikke-nul elementer i søjler, som side j linker til.
    # Hvis side j ikke linker til nogen, får alle elementer i række j værdien 1/N.
    # E: np.ones([N,N])

    M_d = (((1-d)/N) * E_N) + d*(L.T)
    return M_d

In [4]:
def eigenvector_PageRank(web,d=0.85):
        # Input: web er en ordbog over websider og links.
        # d er dæmpningen
        # Output: En ordbog med de samme nøgler som web og værdierne er PageRank for nøglerne

        ranking = dict()
        M = modified_link_matrix(web, d=d)
        eigenvalues, eigenvects = np.linalg.eig(M)

        idx = np.argmax(np.abs(eigenvalues))
        v = eigenvects[:,idx]/np.sum(eigenvects[:,idx])

        for i, page in enumerate(web.keys()):
                ranking[page] = v[i].real  # .real fjerner den imaginære del (0j)

        return ranking

eigenvector_PageRank(W1)

{0: 0.1329106202905259,
 1: 0.24908976517485165,
 2: 0.13668134692273656,
 3: 0.18605748349857465,
 4: 0.2952607841133112}

# Opgave 24

In [5]:
def make_web(n,k,kmin=0):

    assert(k < n), "k skal være mindre end n (da man ikke kan linke til sig selv)"
    assert(kmin <= k), "kmin skal være mindre end eller lig med k"

    keys = [i for i in range(n)]# laver en liste fra [0, 1, 2, ..., n-1]
    web = dict()

    for j in keys:
        numlinks = np.random.randint(kmin,k)
        web[j] = set() 
        links = []
        for _ in range(numlinks): 
            link = np.random.choice(keys) 
            if link != j: 
                links.append(int(link))
        web[j] = set(links) 
    return web

# Opgave 25

In [6]:
import time
web = make_web(5000,10,0) # Vi laver et web med 5000 sider, hvor hver side linker til mellem 0 og 10 andre sider
start = time.time() # antal sekunder siden 1. januar 1970
Pagerank = eigenvector_PageRank(web) # Vi beregner Pagerank for det web vi lige har lavet
time.time() - start  # sekunder brugt på at beregne Pageranken


48.079843044281006

# Opgave 35

In [4]:
def matrix_PageRank(web, power, d=0.85):

    #Input: web er en dictionary hvor nøglerne er websider og værdierne er lister af links til andre websider.
    #d er en positiv float, dæmpningskonstanten og er typisk sat til 0.85, hvilket betyder at der er 85% chance for at følge et link og 15% chance for at hoppe til en tilfældig side.
    #power bestemmer hvor mange gange matrixen ganges med sig selv.
    #Output: Et dictionary med PageRank for hver node.

    #Her opretter jeg et dictionary til at gemme PageRank scores for hver webside.
    ranking = dict()

    #Lav den modificerede link-matrix med dæmpningsfaktoren d.
    M = modified_link_matrix(web, d)

    # Start med den modificerede matrix.
    #Dette gør jeg ved at initialisere M_power til M, og derefter gange den med sig selv power gange i alt.
    #Derefter ganges matrixen med sig selv power antal gange og pageranken findes derefter ved at tage gennemsnittet af hver række.
    M_power = M
    for i in range(power - 1):
        M_power = M_power @ M

    scores = M_power.mean(axis=1)

    #Gem scores i et dictionary med samme nøgler som web.
    for page, score in zip(web.keys(), scores):
        ranking[page] = score

    return ranking

# Opgave 39

In [ ]:
import networkx as nx
import matplotlib.pyplot as plt

def visualize_pagerank(web, ranking):
    # Input: web (dict), ranking (dict med PageRank-værdier)
    # Output: graf hvor pile viser links og nodestørrelser viser PageRank

    G = nx.DiGraph()
    for node, links in web.items():
        G.add_node(node)
        for link in links:
            G.add_edge(node, link)

    sizes = [ranking[n] * 10000 for n in G.nodes()]
    labels = {n: f"{n}\n{ranking[n]:.3f}" for n in G.nodes()}

    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos,
            labels=labels,
            node_color='lightblue',
            node_size=sizes,
            font_size=9,
            arrows=True,
            arrowsize=20,
            edge_color='gray')
    plt.title("PageRank-visualisering")
    plt.show()

visualize_pagerank(W1, eigenvector_PageRank(W1))
visualize_pagerank(W2, eigenvector_PageRank(W2))


NameError: name 'W1' is not defined